### Include Library

In [1]:
from datetime import datetime
import os

# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    FEWSHOT_DEDUP_MESSAGES,
    FEWSHOT_RECALL_MESSAGES,
    FEWSHOT_PRECISION_MESSAGES,
)

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

### 1. build API + processor (inject few-shot examples if you want)
currently fewshot example is at fewshot_examples.py

In [2]:
# 1) build API + processor (inject few-shot examples if you want)
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=FEWSHOT_DEDUP_MESSAGES,           # or None
    fewshot_recall=FEWSHOT_RECALL_MESSAGES,         # or None
    fewshot_precision=FEWSHOT_PRECISION_MESSAGES,   # or None
)


### 2. Load Data

In [3]:
print("Loading caption dataset...")

# number of data points testing
LIMIT = 1

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

#create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

# features that we need to extract from the original dataset
org_caption_dataset = ResultsRepo.read_json("data/low-quality_evaluation_5432-images_2025-04-10_15_29.json")
org_caption_dataset = org_caption_dataset[:LIMIT]

Loading caption dataset...


### 3. generate atomics

In [4]:
# 3) generate atomics
print("Generating atomic statements using gpt-4o...")
T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(org_caption_dataset, limit=LIMIT)

# 3.1) save intermediate
print("Saving intermediate results...")
all_human_captions = []
for item in org_caption_dataset:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)
ResultsRepo.save_results_json(    
    output_path=f"{folder_path}/intermediate_{timestamp}.json",
    org_dataset=org_caption_dataset, 
    T_atomics=T_atomics, 
    g_atomics=g_atomics, 
    parsed_T= parsed_T, 
    T_org=all_human_captions, 
    limit=LIMIT
)

Generating atomic statements using gpt-4o...


100%|██████████| 1/1 [00:12<00:00, 12.75s/it]

Saving intermediate results...
Saved JSON to: results/2025-08-10_01-41/intermediate_2025-08-10_01-41.json


### 4. evaluate and get recall and precision
- match human caption to model caption
- create recall and precision data

In [5]:
# before calculating F1 score, match sentences between human generated and model generated
print("Evaluating atomic statements...")
eval_out = proc.evaluate_matching(all_human_captions, T_atomics, g_atomics)

# 4.1) save evaluation results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/eval_{timestamp}.json",
    update_existing=f"{folder_path}/intermediate_{timestamp}.json",
    metadata=eval_out, 
    limit=LIMIT
)

Evaluating atomic statements...


100%|██████████| 1/1 [00:22<00:00, 22.85s/it]

Saved JSON to: results/2025-08-10_01-41/eval_2025-08-10_01-41.json


### 5. calculate cap f1 score


In [6]:
# 5) calculate cap f1 score
cap_scores = proc.calculate_cap_f1(eval_out)

# 5.1) save cap f1 score results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/final_{timestamp}.json",
    update_existing=f"{folder_path}/eval_{timestamp}.json",
    evaluations=cap_scores, 
    limit=LIMIT
)

100%|██████████| 1/1 [00:00<00:00, 14217.98it/s]

Saved JSON to: results/2025-08-10_01-41/final_2025-08-10_01-41.json


In [7]:
# 6) Final JSON → CSV
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

Saving final results into csv...
CSV file saved to: results/2025-08-10_01-41/final_2025-08-10_01-41.csv
